Tabela tempo x etapa - Maua - planejamento urbano

In [1]:
import pandas as pd
import re
import unicodedata

# Paths Fabric Lakehouse
BASE_SILVER = "/lakehouse/default/Files/silver_planejamento_urbano/"
NOME_TABELA_GOLD = "gold_maua_pl_urbano_etapas"

StatementMeta(, 69083316-c0f6-4570-b962-1112e06ba531, 3, Finished, Available, Finished, False)

In [2]:
%run nb_utils_maua_ingest_acto_gestao

StatementMeta(, 69083316-c0f6-4570-b962-1112e06ba531, 5, Finished, Available, Finished, True)

✓ Funções carregadas: Extração (API) + Transformação (Limpeza)


In [3]:
# Carregar silver_etapas
df_etapas_raw = pd.read_parquet(BASE_SILVER + "silver_maua_pl_urbano_etapas.parquet")
print(f"✓ Carregado: {df_etapas_raw.shape[0]:,} linhas × {df_etapas_raw.shape[1]} colunas")

StatementMeta(, 69083316-c0f6-4570-b962-1112e06ba531, 6, Finished, Available, Finished, False)

✓ Carregado: 146,259 linhas × 15 colunas


In [4]:
# Pipeline de transformação
df_etapas = df_etapas_raw.copy()
df_etapas = tratar_nome_colunas(df_etapas)
df_etapas = colunas_para_snake_case(df_etapas)
df_etapas = converter_colunas_data(df_etapas)

# Renomear seqFluxo para no_solicitacao (padrão em gold_maua_pl_urbano)
# Após snake_case, "seqFluxo" vira "seq_fluxo"
if 'seqfluxo' in df_etapas.columns:
    df_etapas = df_etapas.rename(columns={'seqfluxo': 'no_solicitacao'})

# Dropar colunas esparsas (preserva no_solicitacao automaticamente)
df_etapas = dropar_colunas_esparsas(df_etapas, limite_nan_pct=0.95)

print(f"✓ Transformação concluída: {df_etapas.shape[0]:,} linhas × {df_etapas.shape[1]} colunas")
print(f"✓ Coluna de chave 'no_solicitacao' pronta para relacionamento com gold_maua_pl_urbano")

StatementMeta(, 69083316-c0f6-4570-b962-1112e06ba531, 7, Finished, Available, Finished, False)

✓ Transformação concluída: 146,259 linhas × 15 colunas
✓ Coluna de chave 'no_solicitacao' pronta para relacionamento com gold_maua_pl_urbano


In [5]:
df_etapas

StatementMeta(, 69083316-c0f6-4570-b962-1112e06ba531, 8, Finished, Available, Finished, False)

,codetapa,etapa,servico,no_solicitacao,datacriacaoos,datafinalizacaoos,dataetapainicio,dataetapafim,dataatenderetapa,tempoexecucao,tempoexecucaohoras,status,executor,notifications,isvalid
0,0,RESP. TÉCNICO - PROJETO,ALVARÁ DE APROVAÇÃO DE PROJETO E EXECUÇÃO PARA...,421820,2023-07-27 17:18:29,2024-11-07 10:58:49,2023-07-27 17:18:30,2024-11-07 10:58:49,2023-07-27 17:18:30,468 dias 17 horas 40 minutos 19 segundos,11249.671944,CANCELADA,KIMBERLLY SALTI LUIZ SASAKI,[],True
1,0,BOAS VINDAS,ALVARÁ DE APROVAÇÃO DE PROJETO E EXECUÇÃO PARA...,421820,2023-07-27 17:18:29,2024-11-07 10:58:49,2023-07-27 17:18:29,2023-07-27 17:18:29,2023-07-27 17:18:29,,0.000000,CANCELADA,KIMBERLLY SALTI LUIZ SASAKI,[],True
2,0,RESP. TÉCNICO - PROJETO,ALVARÁ DE APROVAÇÃO DE PROJETO E EXECUÇÃO PARA...,422177,2023-07-28 18:18:48,2024-03-12 10:29:09,2023-07-28 18:18:48,2023-10-02 13:57:55,2023-07-28 18:18:48,65 dias 19 horas 39 minutos 7 segundos,1579.651944,FINALIZADA,FABRICIO DE FREITAS CARDOSO,[],True
3,0,DOCUMENTOS E INFORMAÇÕES NECESSÁRIAS,ALVARÁ DE APROVAÇÃO DE PROJETO E EXECUÇÃO PARA...,422177,2023-07-28 18:18:48,2024-03-12 10:29:09,2023-10-02 13:57:55,2024-03-12 08:50:54,2023-10-02 13:57:55,161 dias 18 horas 52 minutos 59 segundos,3882.883056,FINALIZADA,FABRICIO DE FREITAS CARDOSO,[],True
4,0,BOAS VINDAS,ALVARÁ DE APROVAÇÃO DE PROJETO E EXECUÇÃO PARA...,422177,2023-07-28 18:18:48,2024-03-12 10:29:09,2023-07-28 18:18:48,2023-07-28 18:18:48,2023-07-28 18:18:48,,0.000000,FINALIZADA,FABRICIO DE FREITAS CARDOSO,[],True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146254,0,BOAS VINDAS,"TRANSFERÊNCIA, BAIXA E ASSUNÇÃO DE RESPONSABIL...",850791,2025-10-22 13:22:54,2025-11-21 13:23:06,2025-10-22 13:22:54,2025-10-22 13:22:54,2025-10-22 13:22:54,,0.000000,CANCELADA,WELLINGTON RAMOS DA SILVA,[],True
146255,0,DOCUMENTOS E INFORMAÇÕES NECESSÁRIAS,"TRANSFERÊNCIA, BAIXA E ASSUNÇÃO DE RESPONSABIL...",851897,2025-10-24 07:58:32,2025-11-23 07:58:45,2025-10-24 07:58:32,2025-11-23 07:58:43,2025-11-23 07:58:43,30 dias 0 hora 0 minuto 11 segundos,720.003056,CANCELADA,WELLINGTON RAMOS DA SILVA,[],True
146256,0,BOAS VINDAS,"TRANSFERÊNCIA, BAIXA E ASSUNÇÃO DE RESPONSABIL...",851897,2025-10-24 07:58:32,2025-11-23 07:58:45,2025-10-24 07:58:32,2025-10-24 07:58:32,2025-10-24 07:58:32,,0.000000,CANCELADA,WELLINGTON RAMOS DA SILVA,[],True
146257,0,DOCUMENTOS E INFORMAÇÕES NECESSÁRIAS,"TRANSFERÊNCIA, BAIXA E ASSUNÇÃO DE RESPONSABIL...",863695,2025-11-19 14:30:00,2025-12-19 14:30:13,2025-11-19 14:30:02,2025-12-19 14:30:13,2025-12-19 14:30:12,30 dias 0 hora 0 minuto 11 segundos,720.003056,CANCELADA,GUILHERME OTAVIO DE SOUZA MOURA,[],True


In [7]:
# Salvar como tabela gold no Lakehouse
spark_df = spark.createDataFrame(df_etapas)
# Remover colunas problemáticas
spark_df = spark_df.drop("notifications", "isvalid")
# Salvar como tabela Gold
spark_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(NOME_TABELA_GOLD)
# Validação
total = spark.sql(f"SELECT COUNT(*) as count FROM {NOME_TABELA_GOLD}").collect()[0]["count"]
print(f"✓ Tabela '{NOME_TABELA_GOLD}' criada com {total:,} registros")

StatementMeta(, 69083316-c0f6-4570-b962-1112e06ba531, 10, Finished, Available, Finished, False)

✓ Tabela 'gold_maua_pl_urbano_etapas' criada com 146,259 registros


In [8]:
# Validar coluna no_solicitacao para relacionamento com gold_maua_pl_urbano
if 'no_solicitacao' in df_etapas.columns:
    print(f"✓ Coluna de chave para relacionamento: 'no_solicitacao'")
    print(f"✓ Valores únicos: {df_etapas['no_solicitacao'].nunique():,}")
    print(f"✓ Nulos: {df_etapas['no_solicitacao'].isna().sum():,}")
    print(f"\n✓ Use esta coluna para fazer JOIN com gold_maua_pl_urbano:")
    print(f"  ON gold_etapas.no_solicitacao = gold_solicitacoes.no_solicitacao")
    
    # Amostra de dados para validação
    display(df_etapas[['no_solicitacao', 'etapa', 'servico', 'status']].head(10))
else:
    print("⚠️ ERRO: Coluna 'no_solicitacao' NÃO encontrada!")

StatementMeta(, 69083316-c0f6-4570-b962-1112e06ba531, 11, Finished, Available, Finished, False)

✓ Coluna de chave para relacionamento: 'no_solicitacao'
✓ Valores únicos: 8,075
✓ Nulos: 0

✓ Use esta coluna para fazer JOIN com gold_maua_pl_urbano:
  ON gold_etapas.no_solicitacao = gold_solicitacoes.no_solicitacao


SynapseWidget(Synapse.DataFrame, 1b71a1a4-03bc-411b-bf2a-f89bdb82583d)